In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from config import tcr_embeddings_path, mhc_embeddings_path, peptide_embeddings_path, all_relations_path, train_path, val_path, test_path, CMV_dataset_path
import pickle
import dhg
from typing import List, Tuple, Optional, Union

/home/tarnickil/.conda/envs/mgr_thesis/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
with open(tcr_embeddings_path, "rb") as f:
    tcr_embbedings = pickle.load(f)

with open(mhc_embeddings_path, "rb") as f:
    mhc_embbedings = pickle.load(f)

with open(peptide_embeddings_path, "rb") as f:
    peptide_embbedings = pickle.load(f)
    
with open(all_relations_path, "rb") as f:
    all_relations = pickle.load(f)

with open(CMV_dataset_path, "rb") as f:
    cmv_dataset = pickle.load(f)

In [3]:
map_dict = {"Non-binding":0, "Binding":1}
all_relations["Binding"] = all_relations["Binding"].replace(map_dict)
tcr_embbedings.rename({"Name":"TCR_name"}, axis=1, inplace=True)

/tmp/ipykernel_566714/292328124.py:2: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  all_relations["Binding"] = all_relations["Binding"].replace(map_dict)


In [4]:
# all_relations.drop_duplicates(inplace=True)
# all_relations.reset_index(inplace=True, drop=True)
# all_relations[all_relations.duplicated() == True] # Pusty zbiór

# all_relations.drop(["index"], axis=1, inplace=True)
all_relations.drop(["Donor"], axis=1, inplace=True)
all_relations.drop(["TRAV", "TRAJ", "TRBV", "TRBD", "TRBJ"], axis=1, inplace=True)

kolumna_bin = 'Binding'
kolumna_count = "Count"
kolumna_klucza = 'Name'
pozostale = [col for col in all_relations.columns if col not in [kolumna_klucza, kolumna_bin, kolumna_count]]

# Zbuduj słownik agregacji dynamicznie
agg_dict = {col: (col, 'first') for col in pozostale}

# Dodaj kolumnę binarną
agg_dict[kolumna_bin] = (kolumna_bin, 'max')
agg_dict[kolumna_count] = (kolumna_count, 'max')

# Zastosuj
all_relations = all_relations.groupby(kolumna_klucza, as_index=False).agg(**agg_dict)

# all_relations.to_pickle(all_relations_path)
maska = all_relations.duplicated(subset=['Name'], keep=False)  # oznacza WSZYSTKIE wystąpienia
all_relations[maska].sort_values('Name')

,Name,pMHC,CDR3a,CDR3b,Epitop,TCR_name,HLA_MHC,Binding,Count


In [5]:
all_relations

,Name,pMHC,CDR3a,CDR3b,Epitop,TCR_name,HLA_MHC,Binding,Count
0,A0101_SLEGGGLGY_NC_binder_TCR_1,A0101_SLEGGGLGY_NC_binder,CAVEALTGGGNKLTF,CASSAYTSGPKEQYF,NC_binder,TCR_1,HLA-A0101,0,1
1,A0101_SLEGGGLGY_NC_binder_TCR_10,A0101_SLEGGGLGY_NC_binder,CAVRDKNSGYSTLTF,CASTSADGYGYTF,NC_binder,TCR_10,HLA-A0101,0,1
2,A0101_SLEGGGLGY_NC_binder_TCR_100,A0101_SLEGGGLGY_NC_binder,CAVRDDDYKLSF,CASRFLAGGSHEQYF,NC_binder,TCR_100,HLA-A0101,0,4
3,A0101_SLEGGGLGY_NC_binder_TCR_1000,A0101_SLEGGGLGY_NC_binder,CAVRGYSSASKIIF,CASSHTEETGELFF,NC_binder,TCR_1000,HLA-A0101,0,1
4,A0101_SLEGGGLGY_NC_binder_TCR_10000,A0101_SLEGGGLGY_NC_binder,CALSDPLPLNAGKSTF,CASSERGGTVNEQFF,NC_binder,TCR_10000,HLA-A0101,0,1
...,...,...,...,...,...,...,...,...,...
3347845,NR(B0801)_AAKGRGAAL_NC_binder_TCR_9995,NR(B0801)_AAKGRGAAL_NC_binder,CALSLDNYGQNFVF,CASSETSLYEQYF,NC_binder,TCR_9995,HLA-B0801,0,1
3347846,NR(B0801)_AAKGRGAAL_NC_binder_TCR_9996,NR(B0801)_AAKGRGAAL_NC_binder,CALSPVNYGQNFVF,CASRDTDTQYF,NC_binder,TCR_9996,HLA-B0801,0,1
3347847,NR(B0801)_AAKGRGAAL_NC_binder_TCR_9997,NR(B0801)_AAKGRGAAL_NC_binder,CALSRIDNYGQNFVF,CASSQEVAGGIGDEQFF,NC_binder,TCR_9997,HLA-B0801,0,1
3347848,NR(B0801)_AAKGRGAAL_NC_binder_TCR_9998,NR(B0801)_AAKGRGAAL_NC_binder,CALSVDNYGQNFVF,CASSYSGTGGGLAGELFF,NC_binder,TCR_9998,HLA-B0801,0,1


In [6]:
z = all_relations[["Epitop", "Binding"]].groupby(["Epitop"]).agg("sum")
z["count"] = all_relations[["Epitop", "Binding"]].groupby(["Epitop"]).agg("count").values
z["pct"] = z["Binding"]/z["count"]

In [7]:
CMV_dane_oczyszczone = all_relations[(all_relations["Epitop"] == "IE-1_CMV_binder") & (all_relations["HLA_MHC"].isin(["HLA-A0301","HLA-A2402"]))]
CMV_dane_oczyszczone.to_pickle(CMV_dataset_path)

In [8]:
t = cmv_dataset["Binding"].value_counts().values
t[1]/t[0]

np.float64(0.11587560829278049)

In [9]:
cmv_dataset


,Name,pMHC,CDR3a,CDR3b,Epitop,TCR_name,HLA_MHC,Binding,Count
2075667,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_1,A0301_KLGGALQAK_IE-1_CMV_binder,CAVEALTGGGNKLTF,CASSAYTSGPKEQYF,IE-1_CMV_binder,TCR_1,HLA-A0301,0,1
2075668,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_10,A0301_KLGGALQAK_IE-1_CMV_binder,CAVRDKNSGYSTLTF,CASTSADGYGYTF,IE-1_CMV_binder,TCR_10,HLA-A0301,0,1
2075669,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_100,A0301_KLGGALQAK_IE-1_CMV_binder,CAVRDDDYKLSF,CASRFLAGGSHEQYF,IE-1_CMV_binder,TCR_100,HLA-A0301,0,4
2075670,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_1000,A0301_KLGGALQAK_IE-1_CMV_binder,CAVRGYSSASKIIF,CASSHTEETGELFF,IE-1_CMV_binder,TCR_1000,HLA-A0301,0,1
2075671,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_10000,A0301_KLGGALQAK_IE-1_CMV_binder,CALSDPLPLNAGKSTF,CASSERGGTVNEQFF,IE-1_CMV_binder,TCR_10000,HLA-A0301,0,1
...,...,...,...,...,...,...,...,...,...
2477404,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9995,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSLDNYGQNFVF,CASSETSLYEQYF,IE-1_CMV_binder,TCR_9995,HLA-A2402,0,1
2477405,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9996,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSPVNYGQNFVF,CASRDTDTQYF,IE-1_CMV_binder,TCR_9996,HLA-A2402,0,1
2477406,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9997,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSRIDNYGQNFVF,CASSQEVAGGIGDEQFF,IE-1_CMV_binder,TCR_9997,HLA-A2402,0,1
2477407,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9998,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSVDNYGQNFVF,CASSYSGTGGGLAGELFF,IE-1_CMV_binder,TCR_9998,HLA-A2402,0,1


In [10]:
Data_all = cmv_dataset[['Name', 'TCR_name', 'HLA_MHC', 'Epitop']]
Data = Data_all.copy()
len(Data['TCR_name'].value_counts().index)

66957

In [11]:
nrow = Data.shape[0]
test_nrow = np.round(nrow*0.2, 0)
val_nrow = np.round(nrow*0.2, 0)
TCRs = np.unique(Data["TCR_name"].values)
rng = np.random.default_rng(seed=42)
test_tcr = np.array([])
val_tcr = np.array([])

while test_nrow >= 0:
    tcr_i = rng.choice(TCRs, size=100)
    print(tcr_i)
    val = np.sum(Data['TCR_name'][Data["TCR_name"].isin(tcr_i)].value_counts().values)
    test_nrow -= val
    test_tcr = np.append(test_tcr, tcr_i)
test_tcr.ravel()

Data = Data[~Data["TCR_name"].isin(test_tcr)]
TCRs = np.unique(Data["TCR_name"].values)
while val_nrow >= 0:
    tcr_i = rng.choice(TCRs, size=100)
    print(tcr_i)
    val = np.sum(Data['TCR_name'][Data["TCR_name"].isin(tcr_i)].value_counts().values)
    val_nrow -= val
    val_tcr = np.append(val_tcr, tcr_i)
val_tcr.ravel()

['TCR_15376' 'TCR_56638' 'TCR_49444' 'TCR_36445' 'TCR_36092' 'TCR_61739'
 'TCR_15177' 'TCR_52021' 'TCR_22138' 'TCR_15673' 'TCR_41724' 'TCR_8529'
 'TCR_54335' 'TCR_55866' 'TCR_53234' 'TCR_57368' 'TCR_40926' 'TCR_17719'
 'TCR_60602' 'TCR_37139' 'TCR_4015' 'TCR_32342' 'TCR_20999' 'TCR_65847'
 'TCR_57097' 'TCR_488' 'TCR_34248' 'TCR_5958' 'TCR_42867' 'TCR_36719'
 'TCR_37143' 'TCR_23692' 'TCR_15550' 'TCR_43418' 'TCR_63503' 'TCR_13844'
 'TCR_6172' 'TCR_59873' 'TCR_26676' 'TCR_48063' 'TCR_19956' 'TCR_55682'
 'TCR_52211' 'TCR_31361' 'TCR_14090' 'TCR_8232' 'TCR_36856' 'TCR_63819'
 'TCR_50850' 'TCR_56905' 'TCR_55791' 'TCR_21727' 'TCR_31928' 'TCR_38123'
 'TCR_39997' 'TCR_12637' 'TCR_42935' 'TCR_19296' 'TCR_54796' 'TCR_51159'
 'TCR_65591' 'TCR_5488' 'TCR_32092' 'TCR_804' 'TCR_34757' 'TCR_29633'
 'TCR_64567' 'TCR_32321' 'TCR_14599' 'TCR_38295' 'TCR_57948' 'TCR_21415'
 'TCR_37897' 'TCR_17828' 'TCR_51366' 'TCR_38665' 'TCR_29897' 'TCR_23672'
 'TCR_44015' 'TCR_50361' 'TCR_66663' 'TCR_36341' 'TCR_19682' 

array(['TCR_65135', 'TCR_24101', 'TCR_28104', ..., 'TCR_63401',
       'TCR_35052', 'TCR_13712'], shape=(13500,), dtype=object)

In [12]:
print(test_tcr)
# array(['TCR_15376', 'TCR_56638', 'TCR_49444', ..., 'TCR_18404',
    #    'TCR_33660', 'TCR_24011'], shape=(20200,), dtype=object)
print(val_tcr)    
# 'TCR_37164' 'TCR_849' 'TCR_55986' ... 'TCR_37826' 'TCR_1864' 'TCR_42771']

not_train_tcr = np.append(val_tcr, test_tcr)

['TCR_15376' 'TCR_56638' 'TCR_49444' ... 'TCR_53494' 'TCR_44475'
 'TCR_43543']
['TCR_65135' 'TCR_24101' 'TCR_28104' ... 'TCR_63401' 'TCR_35052'
 'TCR_13712']


In [13]:
train_tcr = Data_all[~Data_all["TCR_name"].isin(not_train_tcr)]["TCR_name"].value_counts().index
# print(len(train_tcr) + len(val_tcr) + len(test_tcr))

# print(np.sum(np.isin(val_tcr, test_tcr)))
# print(np.sum(np.isin(train_tcr, test_tcr)))
# print(np.sum(np.isin(train_tcr, val_tcr)))
# print(np.sum(np.isin(train_tcr, not_train_tcr)))

print(len(np.unique(np.append(train_tcr, not_train_tcr))))
print(len(Data_all["TCR_name"].value_counts().index))

66957
66957


In [14]:
Train_relations = cmv_dataset[cmv_dataset["TCR_name"].isin(train_tcr)]
Train_relations.reset_index(inplace=True, drop=True)
Validation_relations = cmv_dataset[cmv_dataset["TCR_name"].isin(val_tcr)]
Validation_relations.reset_index(inplace=True, drop=True)
Test_relations = cmv_dataset[cmv_dataset["TCR_name"].isin(test_tcr)]
Test_relations.reset_index(inplace=True, drop=True)
print(Train_relations.shape, Validation_relations.shape, Test_relations.shape, cmv_dataset.shape)

(85588, 9) (23968, 9) (24358, 9) (133914, 9)


In [15]:
Train_relations

,Name,pMHC,CDR3a,CDR3b,Epitop,TCR_name,HLA_MHC,Binding,Count
0,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_100,A0301_KLGGALQAK_IE-1_CMV_binder,CAVRDDDYKLSF,CASRFLAGGSHEQYF,IE-1_CMV_binder,TCR_100,HLA-A0301,0,4
1,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_1000,A0301_KLGGALQAK_IE-1_CMV_binder,CAVRGYSSASKIIF,CASSHTEETGELFF,IE-1_CMV_binder,TCR_1000,HLA-A0301,0,1
2,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_10000,A0301_KLGGALQAK_IE-1_CMV_binder,CALSDPLPLNAGKSTF,CASSERGGTVNEQFF,IE-1_CMV_binder,TCR_10000,HLA-A0301,0,1
3,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_10001,A0301_KLGGALQAK_IE-1_CMV_binder,CALSEAGPDTNAGKSTF,CATSDFGRTGELFF,IE-1_CMV_binder,TCR_10001,HLA-A0301,0,1
4,A0301_KLGGALQAK_IE-1_CMV_binder_TCR_10002,A0301_KLGGALQAK_IE-1_CMV_binder,CALSEAGTNAGKSTF,CASSSVLRAGTLTGANVLTF,IE-1_CMV_binder,TCR_10002,HLA-A0301,0,1
...,...,...,...,...,...,...,...,...,...
85583,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9990,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSEVNYGQNFVF,CSGARWKSSYNEQFF,IE-1_CMV_binder,TCR_9990,HLA-A2402,0,1
85584,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9992,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSEVYGQNFVF,CASNAGTGDYQPQHF,IE-1_CMV_binder,TCR_9992,HLA-A2402,0,1
85585,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9993,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSGDNYGQNFVF,CASSETGTYEQYF,IE-1_CMV_binder,TCR_9993,HLA-A2402,0,1
85586,A2402_AYAQKIFKI_IE-1_CMV_binder_TCR_9995,A2402_AYAQKIFKI_IE-1_CMV_binder,CALSLDNYGQNFVF,CASSETSLYEQYF,IE-1_CMV_binder,TCR_9995,HLA-A2402,0,1


In [16]:
Train_relations.to_pickle(train_path)
Validation_relations.to_pickle(val_path)
Test_relations.to_pickle(test_path)